# Reviewing and Hand-Keying Local References to Lynching Victims

Below you'll see my process for reviewing and hand-keying the mined results of 02_identify_local_lynch_coverage.ipynb. It would be helpful to review that notebook, and the one that preceded it: 01_compile_sundown_county_data.ipynb. But if you're pressed for time, I can summarize the whole process as this: isolating newspapers from counties where lynchings occurred as a way to extract reports of racial violence at the local level. The whole project relies on centuries' long efforts to compile lynching data. It also relies on digitized newspaper archives. The larger goal is to build a pipeline to demarcate sundown towns and counties using computational newspaper analysis–to demarcate their phenomenological and epistemological dimensions as expressed through newspapers.

Anyway, in this notebook, I'm hand-keying the data from my text-mining process. I have 1,268 potential references to local lynchings from 115 separate cases. These potential references were identified by localized searches for victim names (i.e., a keyword search). This is a simple information extraction method, meaning there is undoubtedly false positives in my 1,269 potential references, so I'm reviewing them all before I use them for downstream tasks.

It's not just false positives, though. I also have to determine article boundaries by hand if I want to ensure they are accurate. If you're familiar with Chronicling America (the digitized archive I'm pulling data from), you'll know it doesn't demarcate newspaper articles; it just gives you the full-text of each page. I can guesstimate article length by word count (see how I got 'clippings' in my previous notebook), but to make sure I get accurate article length each time, I need to review things by hand.

So, that's what I'm doing in this notebook, but to make things easier–easier than reading over a .csv file in Excel–I decided to build a little tagging dashboard. This speeds up the work, making it easier to tag data and demarcate full articles. I also decided to go the dashboard tagging route after prompting Codex to build a potential dashboard for me using Python, ipywidgets, and display. I didn't think it would get it right with one-shot prompting, but I was pleasantly surprised! I didn't save the original prompt, unfortunately, but it was something like this:

"I need to build a simple dashboard for tagging rows as either 'verified' or 'unverifiable' in my victim_mention_results.csv. I'd like to use ipywidgets and IPython display if possible. I want the dashboard to render the  contents of the columns: 'victim', 'clipping', and 'chron_am_url'. I also want the dashboard to render a text box where I can copy and paste full articles once they're identified. If I paste an article and select 'verified', I want the respective text saved to the given row in a new column called 'verified_articles'. If I select 'unverifiable, I want to move onto the next row. This dashboard should save each entry as I go so I can stop and return to it at different intervals."

It was about that detailed, I think I'm including everything I had initially asked for in the original prompt. In any case, though, it worked! Codex suggested the following Python libraries:


In [ ]:
import os
import html
import pandas as pd
import ipywidgets as w
from IPython.display import display

And it generated the dashboard code. I reviewed it carefully and found it to be acceptable. Hooray!

In [ ]:
# ---------------------------
# Config
# ---------------------------
DATA_PATH = "results_df_progress.csv"
REQUIRED_COLS = ["victim", "clipping", "chron_am_url"]
TARGET_COL = "verified_articles"
STATUS_COL = "verification_status"  # values: pending, verified, unverifiable


# ---------------------------
# Helpers
# ---------------------------
def atomic_save(_df, path):
    tmp = path + ".tmp"
    _df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def first_unverified_pos(_df):
    idx = _df.index[_df[STATUS_COL].eq("pending")]
    return int(idx[0]) if len(idx) else 0


# ---------------------------
# Load / initialize dataframe
# ---------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Create it in the earlier notebook before running this review UI."
    )

df = pd.read_csv(DATA_PATH)

for c in REQUIRED_COLS:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.reset_index(drop=True)

if TARGET_COL not in df.columns:
    df[TARGET_COL] = pd.Series(pd.NA, index=df.index, dtype="object")
else:
    df[TARGET_COL] = df[TARGET_COL].where(pd.notna(df[TARGET_COL]), pd.NA)

if STATUS_COL not in df.columns:
    has_text = df[TARGET_COL].fillna("").astype(str).str.strip().ne("")
    df[STATUS_COL] = "pending"
    df.loc[has_text, STATUS_COL] = "verified"
else:
    valid = {"pending", "verified", "unverifiable"}
    df[STATUS_COL] = df[STATUS_COL].where(df[STATUS_COL].isin(valid), "pending")

atomic_save(df, DATA_PATH)

state = {"pos": first_unverified_pos(df)}


# ---------------------------
# Widgets
# ---------------------------
progress = w.IntProgress(min=0, max=len(df), value=0, description="Done")
row_label = w.HTML()
victim_html = w.HTML()
clipping_html = w.HTML()
url_html = w.HTML()

decision = w.ToggleButtons(
    options=[("Verified", "verified"), ("Unverifiable", "unverifiable")],
    description="Status:",
    value="verified"
)

entry = w.Textarea(
    value="",
    placeholder="Paste verified article text here...",
    layout=w.Layout(width="100%", height="220px")
)

status = w.HTML()

btn_prev = w.Button(description="Previous")
btn_save = w.Button(description="Save")
btn_save_next = w.Button(description="Save + Next")
btn_next = w.Button(description="Next")
btn_first_unverified = w.Button(description="First Unverified")


# ---------------------------
# UI logic
# ---------------------------
def render():
    pos = state["pos"]
    row = df.iloc[pos]

    done = df[STATUS_COL].isin(["verified", "unverifiable"]).sum()
    progress.value = int(done)

    row_label.value = f"<b>Row {pos + 1} of {len(df)}</b>"

    victim_text = "" if pd.isna(row["victim"]) else str(row["victim"])
    clipping_text = "" if pd.isna(row["clipping"]) else str(row["clipping"])
    url_text = "" if pd.isna(row["chron_am_url"]) else str(row["chron_am_url"])

    victim_html.value = f"<b>Victim:</b> {html.escape(victim_text)}"
    clipping_html.value = (
        "<b>Clipping:</b>"
        "<div style='white-space:pre-wrap;border:1px solid #ddd;padding:8px;margin-top:4px;'>"
        f"{html.escape(clipping_text)}</div>"
    )
    url_html.value = f'<b>chron_am_url:</b> <a href="{html.escape(url_text)}" target="_blank">{html.escape(url_text)}</a>'

    row_status = row[STATUS_COL] if row[STATUS_COL] in ("verified", "unverifiable") else "pending"
    if row_status == "unverifiable":
        decision.value = "unverifiable"
    else:
        decision.value = "verified"

    entry.value = "" if pd.isna(row[TARGET_COL]) else str(row[TARGET_COL])
    entry.disabled = (decision.value == "unverifiable")

    status.value = ""


def save_current():
    pos = state["pos"]

    if decision.value == "unverifiable":
        df.at[pos, TARGET_COL] = pd.NA
        df.at[pos, STATUS_COL] = "unverifiable"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:green;'>Saved as Unverifiable (confirmed NA).</span>"
        return True

    txt = entry.value.strip()
    if txt:
        df.at[pos, TARGET_COL] = txt
        df.at[pos, STATUS_COL] = "verified"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:green;'>Saved as Verified.</span>"
        return True
    else:
        df.at[pos, TARGET_COL] = pd.NA
        df.at[pos, STATUS_COL] = "pending"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:#b26a00;'>Saved as Pending (blank verified text).</span>"
        return True


def on_decision_change(change):
    if change["name"] == "value":
        entry.disabled = (change["new"] == "unverifiable")

def on_prev(_):
    state["pos"] = max(0, state["pos"] - 1)
    render()

def on_next(_):
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()

def on_save(_):
    save_current()
    render()

def on_save_next(_):
    save_current()
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()

def on_first_unverified(_):
    state["pos"] = first_unverified_pos(df)
    render()


decision.observe(on_decision_change)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_save.on_click(on_save)
btn_save_next.on_click(on_save_next)
btn_first_unverified.on_click(on_first_unverified)

ui = w.VBox([
    progress,
    row_label,
    victim_html,
    clipping_html,
    url_html,
    decision,
    w.HTML("<b>verified_articles</b>"),
    entry,
    w.HBox([btn_prev, btn_save, btn_save_next, btn_next, btn_first_unverified]),
    status
])

render()
display(ui)

As I'm writing this, I'm about 320 rows into tagging. My process looks like this:

1) I read the victim name and the clipping. If the clipping provides enough context to determine that the name reference is NOT related to an instance of racial violence, I mark the row 'unverifiable'. If you're thinking, "but it's not unverifiable–you've verified it is NOT a relevant case!"–yes, I agree, but you can think of "unverifiable" as the catch-all tag for "not relevant." I'm just being lazy and not tweaking the dashboard since I only need the verified cases.
2) If the clipping doesn't provide enough context and/or it intimates that the name reference is an instance of racial violence, I click on the chron_am_url link. This takes me to the Chronicling America page containing the full text. I then read the clipping with a larger context and deduce if it's relevant. If it is relevant, I copy and paste the full text of the article from Chronicling America and label it "verified." If it isn't relevant, I select "unverifiable."

That's pretty much it. This hand-tagging of data isn't exactly speedy. It's actually Chronicling America's unreliable servers that have slowed me down more than anything else, but the good news is that it's working. Of the 320 rows I've tagged so far, 127 of them have been relevant articles. Many of the "unverifiable" rows are also duplicate articles–either articles wherein the victim is mentioned more than once within the article, therefore receiving a row per mention in my data, or instances where the article was reprinted–but I'm not counting these cases because I don't want a bunch of duplicate text in my final, verified results. That's all to say that, so far, it looks like my pipeline for identifying local coverage of racial violence has worked. I'll continue to hand-tag the data, but if close to 50% of my 1,268 rows end up being verified local coverage, I think I'll have enough data for downstream tasks.

One last thing about my process. My dashboard has a progress bar, but I've also been reviewing progress by reading over the .csv file and thinking about what these data can tell me:

In [ ]:
results_df = pd.read_csv('results_df_progress.csv')

results_df

Just some general thoughts and observations:

- Local coverage looks very different from national coverage. The national articles on lynchings I've read (in my earlier project of curating DUSLR) are often telegraphed, brief, and provide very little information about the cases. There are plenty of exceptions, but generally speaking, national coverage is not very informative. Local coverage, however, shows far more interest in the details. Local coverage often begins before a lynching has occurred–it reports of the alleged crimes and incarceration, and sometimes, the social unrest and potential of mob violence. It also often reports on court proceedings, using far more legal language and information before and after mob violence. These aspects of local coverage provide more information but they also platform biased or inciteful reporting to a greater degree than national coverage.
- Local coverage, extracted via my process, provides a time lapse of events. It gives insight into the preceding events, the legal events, the atrocities, and subsequent commentary or legal responses. Read in this way, it's easy to see how these moments of heightened racial violence were the culmination of racist presumptions, ideologies, and reporting. They were not momentary lapses into lawlessness or racial animosity. They were part of a pattern of events, repeating over years and across the geography of the U.S.
- Local coverage was often big news. Many of the articles I'm finding are full-page reports on the front page of the given newspaper. They are often just as interested in the accusations made against victims as they are in the mob violence. Many of the events get coverage over weeks and months, too. They are reported on as developing stories with all manner of information related to the cases included.

I could go on, but I'd better finish tagging. What's next, too? Once I'm finished, I'll either try to expand my pipeline for identifying other forms of racial violence in these sundown county newspapers or jump straight into testing this data as a training set for BERT to identify more reports and cases in other newspapers. I feel like I'm on the right track. More soon.